# BioOperatorEnv — GRPO Training (Colab)

This notebook trains a Qwen 3 (or Qwen 2.5) Instruct model with GRPO via TRL + Unsloth + LoRA against the `BioOperatorEnv` environment.

**Recommended runtime:** Colab T4 / L4 / A100 with GPU enabled.

**What this notebook does:**
1. Clones the BioOperatorEnv repo.
2. Installs Unsloth + TRL + peft + the env's runtime deps.
3. Builds a small GRPO dataset from the env (Stage 1 = easy DO recovery).
4. Loads Qwen 2.5 3B Instruct in 4-bit with a LoRA(rank=16) adapter.
5. Runs 200 GRPO steps and logs reward curves.
6. Saves the LoRA adapter to `checkpoints/qwen3-bioperator-lora`.

## 1. Install + clone

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth==2024.10 trl peft datasets bitsandbytes accelerate wandb
!pip install -q numpy scipy pandas pydantic matplotlib tqdm fastapi uvicorn

In [ ]:
import os, sys
if not os.path.exists('meta_env'):
    !git clone https://github.com/<YOUR-USER>/<YOUR-REPO>.git meta_env
%cd meta_env
sys.path.insert(0, '.')
print('repo:', os.getcwd())

## 2. Smoke test the env loads

In [ ]:
from bioperator_env.env import BioOperatorEnv
env = BioOperatorEnv(task_id='do-recovery-easy', seed=42)
obs = env.reset()
print('time_h:', obs.time_h)
print('alarm:', obs.alarm)
print('measurements:', obs.measurements)

## 3. Build GRPO dataset (256 prompts from easy DO recovery)

In [ ]:
from training.rollout import build_dataset
from datasets import Dataset

rows = build_dataset(num_samples=256, task_ids=['do-recovery-easy'], seed=0)
ds = Dataset.from_list(rows)
print(f'Dataset rows: {len(ds)}')
print('First prompt tail:', rows[0]['prompt'][-200:])

## 4. Load Qwen 2.5 3B Instruct + LoRA via Unsloth

In [ ]:
from unsloth import FastLanguageModel
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.0, bias='none', use_gradient_checkpointing='unsloth', random_state=42,
)

## 5. Configure GRPO and train

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.reward_fn import reward_fn

cfg = GRPOConfig(
    output_dir='checkpoints/qwen3-bioperator-lora',
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=128,
    max_steps=200,
    beta=0.04,
    temperature=0.7,
    logging_steps=5,
    save_steps=50,
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=cfg,
    train_dataset=ds,
)
trainer.train()

## 6. Save adapter and plot reward curve

In [ ]:
import os
os.makedirs('checkpoints/qwen3-bioperator-lora', exist_ok=True)
model.save_pretrained('checkpoints/qwen3-bioperator-lora')
tokenizer.save_pretrained('checkpoints/qwen3-bioperator-lora')
print('Saved adapter to checkpoints/qwen3-bioperator-lora')

In [ ]:
import matplotlib.pyplot as plt
import os
log_history = trainer.state.log_history
rewards = [(h['step'], h.get('reward', None)) for h in log_history if 'reward' in h]
if rewards:
    xs, ys = zip(*rewards)
    plt.figure(figsize=(8,4))
    plt.plot(xs, ys, marker='o')
    plt.xlabel('GRPO step')
    plt.ylabel('Mean reward')
    plt.title('BioOperatorEnv GRPO training reward')
    plt.grid(alpha=0.3)
    os.makedirs('results', exist_ok=True)
    plt.savefig('results/reward_curve.png', dpi=120, bbox_inches='tight')
    plt.show()

## 7. Try the trained agent on a fresh episode

Run the trained model on a held-out seed and look at sample actions.

In [ ]:
import re, json
from bioperator_env.prompt import build_prompt

env = BioOperatorEnv(task_id='do-recovery-easy', seed=999)
obs = env.reset()
for k in range(20):
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=64, temperature=0.5,
                          do_sample=True, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            action = json.loads(m.group(0))
        except Exception:
            action = {'feed_delta_L_h': 0, 'aeration_delta_vvm': 0.0, 'agitation_delta_rpm': 0}
    else:
        action = {'feed_delta_L_h': 0, 'aeration_delta_vvm': 0.0, 'agitation_delta_rpm': 0}
    print(f'step {k:2d}  action={action}  DO%={obs.measurements["dissolved_oxygen_pct"]:.1f}')
    obs, r, done, info = env.step(action)
    if done:
        break
print('Done. Cumulative reward:', env.cumulative_reward)